# Metajeux-03 : L'abstraction a dette mesurable

**Navigation** : [Version Python](Metajeux-03-Abstraction-a-Dette-Python.ipynb) | [Index](README.md)

**Kernel** : .NET C# (`.net-csharp`)

## Le retournement

Face a un jeu trop grand pour etre resolu, on construit un jeu abstrait plus petit, on le resout, **puis on retransporte la strategie vers le jeu original**. Le geste ordinaire s'arrete a "l'abstraction semble bonne". Kroer & Sandholm ne s'y arretent pas : ils bornent la qualite de la solution **apres retour dans le jeu d'origine**.

```
G --alpha--> G_tilde --solve--> sigma_tilde --rho--> sigma_G
       avec        Exploitabilite(sigma_G) <= ε(alpha, rho, ...)
```

D'ou le retournement de la question :

> plus "**cette representation compresse-t-elle ?**"
> mais "**QU'AI-JE LE DROIT D'OUBLIER SANS PERDRE LA PROPRIETE QUI M'INTERESSE ?** »

```
K  <->  abstraction  <->  perte controlee  <->  DETTE DE REPRESENTATION
```

C'est la forme respectable de ce que le depot appelait "passage entre lentilles" -- avec, cette fois, une dette **chiffrable**.

## Ce que ce jumeau recalcule, et ce qu'il cite

Ce carnet est le jumeau C# de `Metajeux-03-Abstraction-a-Dette-Python.ipynb` : meme donnee, meme demonstration, meme plan. Trois frontieres sont ecrites en toutes lettres, parce qu'un jumeau qui les cache n'est plus un jumeau mais une copie qui pretend :

- **RECALCULE en BCL** : l'abstraction, le solve exact par enumeration de supports, le retransport, l'exploitabilite, la courbe de dette, le diagnostic par etat, et les trois valeurs de la dette retransportee (3 / 4 / 0).
- **CITE** : les 24 gains des 6 duels de G. Le jumeau Python les tire d'un generateur `random.seed(42)` ; le MT19937 de Python n'est **pas** reconstructible en BCL, donc ces valeurs sont reprises **verbatim** et etiquetees comme donnees citees -- jamais presentees comme recalculees. Les **invariants** de ces donnees (portee `[-3, 3]`, 24 gains, 6 duels distincts) sont, eux, verifies a l'execution.
- **CITE** : l'exemple guide 4. Ce carnet ne lance aucun processus externe (ni compilateur Lean, ni solveur, ni reseau) et n'importe rien du module `Abstraction` : la sortie du compilateur Lean sur ce module -- theorem nomme, axioms, valeurs reduites par le noyau -- y est reprise comme **donnee de reference**, explicitement marquee. Les fonctions que le carnet consomme dans le module (`gain`, `vHaut`, `vBas`, `dette`, `sommeBloc`, `detteBloc`) sont, elles, **reimplementees en BCL** : le carnet rejoue les duels abstraits et les dettes retransportees, et **confronte** ses trois valeurs (3 / 4 / 0) aux valeurs citees -- un ecart echoue la cellule.

## Le notebook

Quatre **exemples guides** (resolus), chacun suivi d'un **exercice** non resolu, sur un jeu petit mais non trivial (2 joueurs, somme nulle, 6 etats) :

1. **Abstraire** -- fusionner des etats ou des actions ; mesurer la taille gagnee.
2. **Resoudre et relever** -- resoudre dans l'abstrait, retransporter, **mesurer l'exploitabilite dans le jeu d'origine**. C'est tout le point.
3. **La courbe de dette** -- faire varier la grosserete de l'abstraction et tracer taille contre exploitabilite. La forme de cette courbe **est** le livrable.
4. **Le miroir forme** -- le module Lean `Abstraction` certifie un contre-exemple a la lecture naturelle de la courbe ; le carnet en **cite** la preuve formelle et **rejoue en BCL** les trois dettes qu'elle porte (3 / 4 / 0) : la courbe mesuree en C# recoupe une preuve, donc ce n'est pas une loi empirique.

## La frontiere honnete

Les bornes theoriques de Kroer & Sandholm sont **RAPPORTEES** (la digestion elle-meme signale qu'elle schematise le resultat plutot que sa formulation theorematique exacte). Le notebook **mesure** son cas ; il ne redemontre pas la borne. La distinction s'ecrit.

***

## Prerequis

- `GameTheory-13-ImperfectInfo-CFR-Python.ipynb` (information imparfaite)
- Notions de strategie mixte, exploitabilite, regret

## Duree estimee : 35 minutes

***


In [1]:
// Cellule 1 -- Construction du jeu d'origine G (2 joueurs, zero-sum, 6 etats)
//
// TRANCHAGE (ce qu'est G, explicitement) : G est la SOMME de 6 duels 2x2 independants,
// un duel par etat. Dans chaque etat s, chaque joueur choisit une action a in {0, 1}
// (strategie comportementale). Le gain total du joueur 1 est la somme des gains des
// 6 duels ; u2 = -u1 (zero-sum). Ce choix rend tout calculable EXACTEMENT : chaque
// duel 2x2 se resout par enumeration de supports, et G se resout duel par duel.
// C'est le socle sur quoi la dette d'abstraction se mesurera (exemples guides 2 et 3).
//
// DONNEES CITEES : les 24 gains des 6 duels sont ceux du jumeau Python
// (random.seed(42) ; random.randint(-3, 3)), recopies verbatim. Le MT19937 de Python
// n'est PAS reconstructible en BCL : ces valeurs sont donc des DONNEES CITEES -- affichees
// et etiquetees comme telles, jamais presentees comme recalculees ici. Les invariants
// de ces donnees (portee, nombre de gains, unicite des duels) sont, eux, verifies.
// Tout le reste du carnet est RECALCULE en BCL.

using System;
using System.Collections.Generic;
using System.Globalization;

Console.WriteLine("Jeu G : somme de 6 duels 2x2 independants (zero-sum), gains cites (graine Python 42).");
Console.WriteLine("Une strategie est comportementale : une action mixte par etat et par joueur.");
Console.WriteLine();
Console.WriteLine("Les 6 duels (ligne = action du joueur 1, colonne = action du joueur 2) :");
for (int s = 0; s < Mj03Jeu.NStates; s++)
{
    Console.WriteLine($"  M_{s} = {Mj03Jeu.MatriceLigne(s)}");
}

// Verification des invariants des DONNEES CITEES (portee des tirages, comptage, unicite).
int totalGains = 0;
var duelsVus = new HashSet<string>();
for (int s = 0; s < Mj03Jeu.NStates; s++)
{
    for (int a1 = 0; a1 < Mj03Jeu.NActions; a1++)
    {
        for (int a2 = 0; a2 < Mj03Jeu.NActions; a2++)
        {
            int g = Mj03Jeu.Gain(s, a1, a2);
            Mj03Format.That(g >= -3 && g <= 3, $"gain hors de la plage de tirage [-3, 3] dans M_{s} : {g}");
            totalGains++;
        }
    }
    duelsVus.Add(Mj03Jeu.MatriceLigne(s));
}
Mj03Format.That(totalGains == 24, $"24 gains attendus, {totalGains} comptes");
Mj03Format.That(duelsVus.Count == 6, "les 6 duels cites doivent etre distincts");
Console.WriteLine();
Console.WriteLine($"Invariants des donnees citees verifies : {totalGains} gains dans [-3, 3], {duelsVus.Count} duels distincts.");

public static class Mj03Format
{
    // Formatage invariant (le point decimal ne depend pas de la culture de la machine).
    public static string N(double v, int dec) => v.ToString("F" + dec, CultureInfo.InvariantCulture);

    public static void That(bool cond, string msg)
    {
        if (!cond)
        {
            throw new InvalidOperationException("Assertion FAUSSE : " + msg);
        }
    }

    // Les lignes de la sortie du noyau qui ne contiennent qu'un entier : les valeurs
    // reduites par les `#eval` du temoin, dans l'ordre des `#eval`.
    public static List<int> Entiers(string sortie)
    {
        var valeurs = new List<int>();
        foreach (string ligne in sortie.Split('\n'))
        {
            string texte = ligne.Trim();
            int valeur;
            if (texte.Length > 0 && int.TryParse(texte, out valeur))
            {
                valeurs.Add(valeur);
            }
        }
        return valeurs;
    }

    public static void Close(double actual, double expected, double tol, string msg)
    {
        if (Math.Abs(actual - expected) > tol)
        {
            throw new InvalidOperationException(
                $"Assertion FAUSSE : {msg} -- attendu {N(expected, 6)}, obtenu {N(actual, 6)}");
        }
    }
}

public static class Mj03Jeu
{
    public const int NStates = 6;
    public const int NActions = 2;

    // DONNEES CITEES -- voir l'en-tete de la cellule.
    public static readonly int[][] Duels =
    {
        new[] { 2, -3, -3, 2 },
        new[] { -1, -2, -2, -2 },
        new[] { 2, -3, 2, 2 },
        new[] { 1, -3, 1, 0 },
        new[] { -3, -3, -3, -2 },
        new[] { -2, 1, 1, -3 },
    };

    public static int Gain(int s, int a1, int a2) => Duels[s][a1 * NActions + a2];

    public static double[,] Matrice(int s)
    {
        var m = new double[NActions, NActions];
        for (int a1 = 0; a1 < NActions; a1++)
        {
            for (int a2 = 0; a2 < NActions; a2++)
            {
                m[a1, a2] = Gain(s, a1, a2);
            }
        }
        return m;
    }

    public static string MatriceLigne(int s)
    {
        var lignes = new List<string>();
        for (int a1 = 0; a1 < NActions; a1++)
        {
            var cases = new List<string>();
            for (int a2 = 0; a2 < NActions; a2++)
            {
                cases.Add(Gain(s, a1, a2).ToString(CultureInfo.InvariantCulture));
            }
            lignes.Add("[" + string.Join(", ", cases) + "]");
        }
        return "[" + string.Join(", ", lignes) + "]";
    }
}


The below script needs to be able to find the current output cell; this is an easy method to get it.

Jeu G : somme de 6 duels 2x2 independants (zero-sum), gains cites (graine Python 42).


Une strategie est comportementale : une action mixte par etat et par joueur.


Les 6 duels (ligne = action du joueur 1, colonne = action du joueur 2) :


  M_0 = [[2, -3], [-3, 2]]


  M_1 = [[-1, -2], [-2, -2]]


  M_2 = [[2, -3], [2, 2]]


  M_3 = [[1, -3], [1, 0]]


  M_4 = [[-3, -3], [-3, -2]]


  M_5 = [[-2, 1], [1, -3]]


Invariants des donnees citees verifies : 24 gains dans [-3, 3], 6 duels distincts.


## Exemple guide 1 -- Abstraire

**Ce que fait l'exemple** : sur le jeu G a 6 etats, une **abstraction par fusion d'etats**. Concretement :

- Choisir une partition des 6 etats en 3 paires : `{{s0, s1}, {s2, s3}, {s4, s5}}`.
- Definir le jeu abstrait `G_tilde` : chaque bloc devient un **etat abstrait**, muni d'un duel 2x2 **moyenne** des duels fusionnes.
- Mesurer la taille gagnee : `|G| = 6 etats` vs `|G_tilde| = 3 etats abstraits`.

**Ce qu'est G_tilde (tranchage explicite)** : `G_tilde` est un **jeu a etats**, de meme forme que G -- un duel 2x2 (moyenne) par etat abstrait, payoff total = somme. Comme le payoff de G est additif par etat, cette forme a la meme valeur que le jeu a une etape sur **meta-actions** (ou chaque joueur choisit d'un coup son action dans chaque etat abstrait) ; c'est la forme a etats que nous retenons, car la retransportation vers G y est naturelle : l'etat abstrait fournit une action mixte, chaque etat originel du bloc la recopie.

**Sortie attendue** :
- `G_tilde` exhibe en toutes lettres (3 duels moyens 2x2).
- Le facteur de reduction `|G| / |G_tilde| = 2`.

**Note pedagogique** : on n'a pas encore mesure la qualite de la solution. C'est l'exemple guide 2.


In [2]:
// Exemple guide 1 : construire G_tilde (fusion d'etats, duels moyennes)
using System;
using System.Globalization;
using System.Linq;

int[][] partition = { new[] { 0, 1 }, new[] { 2, 3 }, new[] { 4, 5 } };  // 3 blocs de fusion
int nAbstract = partition.Length;

Console.WriteLine($"Jeu abstrait G_tilde : {nAbstract} etats abstraits (un duel 2x2 moyen par bloc), zero-sum.");
for (int k = 0; k < nAbstract; k++)
{
    double[,] m = Mj03Abstraction.AbstractMatrix(k, partition);
    string bloc = "[" + string.Join(", ", partition[k].Select(x => x.ToString(CultureInfo.InvariantCulture))) + "]";
    string rendu = "[[" + Mj03Format.N(m[0, 0], 3) + ", " + Mj03Format.N(m[0, 1], 3) + "], ["
                        + Mj03Format.N(m[1, 0], 3) + ", " + Mj03Format.N(m[1, 1], 3) + "]]";
    Console.WriteLine($"  M~_{k} (bloc {bloc}) = {rendu}");
}

double reduction = (double)Mj03Jeu.NStates / nAbstract;
Console.WriteLine();
Console.WriteLine($"Facteur de reduction : {Mj03Jeu.NStates} etats / {nAbstract} etats abstraits = {Mj03Format.N(reduction, 1)}");

// Parite avec le jumeau Python : les trois duels moyens sont RECALCULES ici, puis
// confrontes aux valeurs exactes du carnet Python (moyennes des memes duels cites).
double[][] attendu =
{
    new[] { 0.5, -2.5, -2.5, 0.0 },
    new[] { 1.5, -3.0, 1.5, 1.0 },
    new[] { -2.5, -1.0, -1.0, -2.5 },
};
for (int k = 0; k < nAbstract; k++)
{
    double[,] m = Mj03Abstraction.AbstractMatrix(k, partition);
    double[] a = attendu[k];
    Mj03Format.Close(m[0, 0], a[0], 1e-9, $"M~_{k}[0,0]");
    Mj03Format.Close(m[0, 1], a[1], 1e-9, $"M~_{k}[0,1]");
    Mj03Format.Close(m[1, 0], a[2], 1e-9, $"M~_{k}[1,0]");
    Mj03Format.Close(m[1, 1], a[3], 1e-9, $"M~_{k}[1,1]");
}
Mj03Format.Close(reduction, 2.0, 1e-12, "facteur de reduction");
Console.WriteLine("Parite avec le jumeau Python : les 3 duels moyens et le facteur de reduction coincident.");

public static class Mj03Abstraction
{
    // Index du bloc (etat abstrait) contenant l'etat originel s.
    public static int AbstractState(int s, int[][] partition)
    {
        for (int k = 0; k < partition.Length; k++)
        {
            if (partition[k].Contains(s))
            {
                return k;
            }
        }
        return -1;  // inaccessible : la partition couvre tous les etats
    }

    // Duel abstrait du bloc k : MOYENNE des duels des etats fusionnes du bloc.
    public static double[,] AbstractMatrix(int k, int[][] partition)
    {
        int[] block = partition[k];
        var m = new double[2, 2];
        for (int a1 = 0; a1 < 2; a1++)
        {
            for (int a2 = 0; a2 < 2; a2++)
            {
                m[a1, a2] = block.Select(s => Mj03Jeu.Matrice(s)[a1, a2]).Average();
            }
        }
        return m;
    }
}


Jeu abstrait G_tilde : 3 etats abstraits (un duel 2x2 moyen par bloc), zero-sum.


  M~_0 (bloc [0, 1]) = [[0.500, -2.500], [-2.500, 0.000]]


  M~_1 (bloc [2, 3]) = [[1.500, -3.000], [1.500, 1.000]]


  M~_2 (bloc [4, 5]) = [[-2.500, -1.000], [-1.000, -2.500]]


Facteur de reduction : 6 etats / 3 etats abstraits = 2.0


Parite avec le jumeau Python : les 3 duels moyens et le facteur de reduction coincident.


## Exercice 1 -- Une autre partition, une autre reduction

L'exemple guide 1 a fusionne les etats en 3 paires regulieres. Une abstraction a **4 etats** existe aussi : la partition **P4** = {0,1} {2} {3} {4,5} -- deux blocs fusionnes, deux singletons.

**Enonce** : construisez le jeu abstrait G_tilde(P4) -- un duel moyen par bloc, les singletons gardant leur duel tel quel -- et affichez les 4 duels abstraits puis le facteur de reduction |G| / |G_tilde(P4)|.

**Sortie attendue** : 4 matrices (blocs + duels moyennees, arrondis a 3 decimales) et un facteur de reduction non entier.

**Indices** :

- `# Indice` : `Mj03Abstraction.AbstractMatrix(k, partition)` et `Mj03Abstraction.AbstractState(s, partition)` acceptent la partition en argument -- l'exemple guide 1 les a definies generiques.
- `# Etape 1` : poser la partition sous la forme d'un tableau de tableaux d'entiers, comme `partition` dans l'exemple guide 1.
- `# Etape 2` : boucler sur les blocs, afficher la matrice moyenne arrondie.
- `# Etape 3` : le facteur de reduction est `Mj03Jeu.NStates / (double)partition.Length`.


In [3]:
// Exercice 1 : G_tilde pour la partition P4 -- a completer
using System;
using System.Globalization;
using System.Linq;

int[][] partition4 = { new[] { 0, 1 }, new[] { 2 }, new[] { 3 }, new[] { 4, 5 } };  // 4 etats abstraits : 2 blocs fusionnes, 2 singletons

// TODO etudiant : afficher les 4 duels abstraits de P4 (matrices moyennees, arrondies)
// puis le facteur de reduction |G| / |G_tilde(P4)|.
// Indice : Mj03Abstraction.AbstractMatrix(k, partition4) fait deja la moyenne d'un bloc.

Console.WriteLine("Exercice a completer : construire G_tilde(P4) et mesurer la reduction.");
object resultatEx1 = null;  // TODO etudiant


Exercice a completer : construire G_tilde(P4) et mesurer la reduction.


## Exemple guide 2 -- Resoudre et relever

**Ce que fait l'exemple** : resoudre le jeu abstrait `G_tilde` **exactement**, retransporter la strategie vers le jeu d'origine (chaque etat originel d'un bloc joue la strategie de l'etat abstrait), puis **mesurer l'exploitabilite de la strategie retransportee DANS le jeu d'origine**.

**Critere d'acceptation** : l'exploitabilite est mesuree dans le jeu d'origine, **jamais** dans l'abstrait. C'est tout le point.

**Resolution exacte, pas d'approximation iterative** : chaque duel 2x2 est resolu par **enumeration de supports** -- recherche d'un point selle pur (4 profils), sinon equilibre a support complet (formule fermee). Les duels de `G_tilde` etant independants (payoff additif), le resoudre bloc par bloc le resout exactement.

**Definition (unique, reutilisee telle quelle par l'exemple guide 3)** -- l'exploitabilite d'une paire de strategies `(sigma_1, sigma_2)` dans un jeu zero-sum est la somme des gains des deux meilleures reponses :

```
expl(sigma_1, sigma_2) = max_{pi_1} u_1(pi_1, sigma_2)  +  max_{pi_2} u_2(sigma_1, pi_2)
```

Elle est **nulle si et seulement si** la paire est un equilibre de Nash. Comme `u_2 = -u_1`, elle se decompose autour de la valeur `v` du jeu :

```
expl = (BR_1 - v) + (BR_2 + v)
```

chaque terme mesurant ce qu'un joueur gagne a devier -- la dette se lit des deux cotes.


In [4]:
// Exemple guide 2 : solve exact de G_tilde, retransport, mesure DANS G
using System;
using System.Collections.Generic;
using System.Globalization;
using System.Linq;

// Resolution EXACTE de G_tilde : un equilibre par etat abstrait.
int[][] partition3 = { new[] { 0, 1 }, new[] { 2, 3 }, new[] { 4, 5 } };
var sigmaTilde = new List<(double[] p, double[] q, double val)>();
for (int k = 0; k < partition3.Length; k++)
{
    sigmaTilde.Add(Mj03Mesures.Solve2x2(Mj03Abstraction.AbstractMatrix(k, partition3)));
}

// Valeur exacte de G : somme des valeurs des 6 duels (ils sont independants).
double valueG = 0;
for (int s = 0; s < Mj03Jeu.NStates; s++)
{
    valueG += Mj03Mesures.Solve2x2(Mj03Jeu.Matrice(s)).val;
}

// Retransport : l'etat originel s recopie la strategie de son bloc.
var (sig1, sig2, _) = Mj03Mesures.SolveAndTransport(partition3);

(double br1, double br2) = Mj03Mesures.BestResponseTerms(Mj03Jeu.Gain, sig1, sig2, Mj03Jeu.NStates);
double expl = Mj03Mesures.Exploitabilite(Mj03Jeu.Gain, sig1, sig2, Mj03Jeu.NStates);

Console.WriteLine($"Valeur exacte de G (somme des 6 duels) : v(G) = {Mj03Format.N(valueG, 4)}");
Console.WriteLine();
Console.WriteLine("Resolution exacte de G_tilde (enumeration de supports, bloc par bloc) :");
for (int k = 0; k < partition3.Length; k++)
{
    (double[] p, double[] q, double val) = sigmaTilde[k];
    string bloc = "[" + string.Join(", ", partition3[k].Select(x => x.ToString(CultureInfo.InvariantCulture))) + "]";
    Console.WriteLine($"  bloc {bloc} : p* = [{Mj03Format.N(p[0], 3)}, {Mj03Format.N(p[1], 3)}]"
        + $"  q* = [{Mj03Format.N(q[0], 3)}, {Mj03Format.N(q[1], 3)}]  val = {Mj03Format.N(val, 3)}");
}
Console.WriteLine();
Console.WriteLine("Mesure DANS G de la strategie retransportee :");
Console.WriteLine($"  BR_1 = max_pi1 u_1(pi_1, sigma_2) = {Mj03Format.N(br1, 4)}"
    + $"   (v(G) = {Mj03Format.N(valueG, 4)}, ecart = {Mj03Format.N(br1 - valueG, 4)})");
Console.WriteLine($"  BR_2 = max_pi2 u_2(sigma_1, pi_2) = {Mj03Format.N(br2, 4)}"
    + $"   (-v(G) = {Mj03Format.N(-valueG, 4)}, ecart = {Mj03Format.N(br2 + valueG, 4)})");
Console.WriteLine($"  expl = (BR_1 - v) + (BR_2 + v) = {Mj03Format.N(br1 - valueG, 4)} + {Mj03Format.N(br2 + valueG, 4)} = {Mj03Format.N(expl, 4)}");
Console.WriteLine();
Console.WriteLine($"Exploitabilite = {Mj03Format.N(expl, 4)} > 0 : la strategie retransportee n'est PAS un equilibre de G.");
Console.WriteLine("La perte causee par l'abstraction est chiffree, des deux cotes du jeu.");

// Parite avec le jumeau Python : valeurs EXACTES (les memes, avant arrondi d'affichage).
Mj03Format.Close(valueG, -4.214285714285714, 1e-9, "v(G)");
Mj03Format.Close(sigmaTilde[0].p[0], 0.45454545454545453, 1e-9, "p* bloc (0,1)");
Mj03Format.Close(sigmaTilde[0].q[0], 0.45454545454545453, 1e-9, "q* bloc (0,1)");
Mj03Format.Close(sigmaTilde[0].val, -1.1363636363636365, 1e-9, "val bloc (0,1)");
Mj03Format.Close(sigmaTilde[1].p[0], 0.0, 1e-9, "p* bloc (2,3)");
Mj03Format.Close(sigmaTilde[1].q[0], 0.0, 1e-9, "q* bloc (2,3)");
Mj03Format.Close(sigmaTilde[1].val, 1.0, 1e-9, "val bloc (2,3)");
Mj03Format.Close(sigmaTilde[2].p[0], 0.5, 1e-9, "p* bloc (4,5)");
Mj03Format.Close(sigmaTilde[2].q[0], 0.5, 1e-9, "q* bloc (4,5)");
Mj03Format.Close(sigmaTilde[2].val, -1.75, 1e-9, "val bloc (4,5)");
Mj03Format.Close(br1, -2.8181818181818183, 1e-9, "BR_1");
Mj03Format.Close(br2, 4.7272727272727275, 1e-9, "BR_2");
Mj03Format.Close(expl, 1.9090909090909092, 1e-9, "exploitabilite");
Console.WriteLine("Parite avec le jumeau Python : v(G), les 3 equilibres de bloc, BR_1, BR_2 et l'exploitabilite coincident.");

public static class Mj03Mesures
{
    // Equilibre EXACT d'un duel 2x2 zero-sum, par ENUMERATION DE SUPPORTS.
    // Retourne (p, q, val) : p[a1] = proba de ligne a1 (joueur 1),
    // q[a2] = proba de colonne a2 (joueur 2), val = valeur du duel.
    public static (double[] p, double[] q, double val) Solve2x2(double[,] m)
    {
        double a = m[0, 0], b = m[0, 1], c = m[1, 0], d = m[1, 1];
        // Supports purs : point selle (maximum de sa colonne ET minimum de sa ligne)
        for (int i = 0; i < 2; i++)
        {
            for (int j = 0; j < 2; j++)
            {
                double maxCol = Math.Max(m[0, j], m[1, j]);
                double minRow = Math.Min(m[i, 0], m[i, 1]);
                if (Math.Abs(m[i, j] - maxCol) < 1e-12 && Math.Abs(m[i, j] - minRow) < 1e-12)
                {
                    double[] p = { i == 0 ? 1.0 : 0.0, i == 1 ? 1.0 : 0.0 };
                    double[] q = { j == 0 ? 1.0 : 0.0, j == 1 ? 1.0 : 0.0 };
                    return (p, q, m[i, j]);
                }
            }
        }
        // Support mixte complet : formule fermee (denominateur non nul car pas de selle)
        double den = a - b - c + d;
        double pStar = (d - c) / den;
        double qStar = (d - b) / den;
        Mj03Format.That(pStar >= 0 && pStar <= 1 && qStar >= 0 && qStar <= 1,
                       "probabilites d'equilibre hors de [0, 1]");
        return (new[] { pStar, 1 - pStar }, new[] { qStar, 1 - qStar }, (a * d - b * c) / den);
    }

    // Les deux termes de l'exploitabilite :
    // (max_pi1 u_1(pi_1, sig_2), max_pi2 u_2(sig_1, pi_2)).
    public static (double br1, double br2) BestResponseTerms(
        Func<int, int, int, int> gain,
        Func<int, int, double> sig1,
        Func<int, int, double> sig2,
        int nStates)
    {
        double br1 = 0;
        double br2 = 0;
        for (int s = 0; s < nStates; s++)
        {
            double maxJ1 = double.NegativeInfinity;
            double maxJ2 = double.NegativeInfinity;
            for (int a1 = 0; a1 < 2; a1++)
            {
                double u1 = 0;
                for (int a2 = 0; a2 < 2; a2++)
                {
                    u1 += sig2(s, a2) * gain(s, a1, a2);
                }
                if (u1 > maxJ1)
                {
                    maxJ1 = u1;
                }
            }
            for (int a2 = 0; a2 < 2; a2++)
            {
                double u2 = 0;
                for (int a1 = 0; a1 < 2; a1++)
                {
                    u2 += sig1(s, a1) * (-gain(s, a1, a2));
                }
                if (u2 > maxJ2)
                {
                    maxJ2 = u2;
                }
            }
            br1 += maxJ1;
            br2 += maxJ2;
        }
        return (br1, br2);
    }

    // expl(sig_1, sig_2) = max_pi1 u_1(pi_1, sig_2) + max_pi2 u_2(sig_1, pi_2).
    // Nulle si et seulement si (sig_1, sig_2) est un equilibre de Nash (zero-sum).
    public static double Exploitabilite(
        Func<int, int, int, int> gain,
        Func<int, int, double> sig1,
        Func<int, int, double> sig2,
        int nStates)
    {
        (double br1, double br2) = BestResponseTerms(gain, sig1, sig2, nStates);
        return br1 + br2;
    }

    // Solve exact de G_tilde pour la partition donnee, puis retransport vers G.
    public static (Func<int, int, double> sig1, Func<int, int, double> sig2, double valueTilde)
        SolveAndTransport(int[][] partition)
    {
        int n = partition.Length;
        var p = new double[n][];
        var q = new double[n][];
        double valueTilde = 0;
        for (int k = 0; k < n; k++)
        {
            (double[] pp, double[] qq, double val) = Solve2x2(Mj03Abstraction.AbstractMatrix(k, partition));
            p[k] = pp;
            q[k] = qq;
            valueTilde += val;
        }
        var where = new int[Mj03Jeu.NStates];
        for (int k = 0; k < n; k++)
        {
            foreach (int s in partition[k])
            {
                where[s] = k;
            }
        }
        Func<int, int, double> sig1 = (s, a) => p[where[s]][a];
        Func<int, int, double> sig2 = (s, a) => q[where[s]][a];
        return (sig1, sig2, valueTilde);
    }

    // Contribution de chaque etat a l'exploitabilite (diagnostic du palier).
    public static double[] ContributionsParEtat(
        Func<int, int, int, int> gain,
        Func<int, int, double> sig1,
        Func<int, int, double> sig2,
        int nStates)
    {
        var contrib = new double[nStates];
        for (int s = 0; s < nStates; s++)
        {
            double br1s = double.NegativeInfinity;
            double br2s = double.NegativeInfinity;
            for (int a1 = 0; a1 < 2; a1++)
            {
                double u1 = 0;
                for (int a2 = 0; a2 < 2; a2++)
                {
                    u1 += sig2(s, a2) * gain(s, a1, a2);
                }
                if (u1 > br1s)
                {
                    br1s = u1;
                }
            }
            for (int a2 = 0; a2 < 2; a2++)
            {
                double u2 = 0;
                for (int a1 = 0; a1 < 2; a1++)
                {
                    u2 += sig1(s, a1) * (-gain(s, a1, a2));
                }
                if (u2 > br2s)
                {
                    br2s = u2;
                }
            }
            contrib[s] = br1s + br2s;
        }
        return contrib;
    }
}


Valeur exacte de G (somme des 6 duels) : v(G) = -4.2143


Resolution exacte de G_tilde (enumeration de supports, bloc par bloc) :


  bloc [0, 1] : p* = [0.455, 0.545]  q* = [0.455, 0.545]  val = -1.136


  bloc [2, 3] : p* = [0.000, 1.000]  q* = [0.000, 1.000]  val = 1.000


  bloc [4, 5] : p* = [0.500, 0.500]  q* = [0.500, 0.500]  val = -1.750


Mesure DANS G de la strategie retransportee :


... (2 outputs left out)

  expl = (BR_1 - v) + (BR_2 + v) = 1.3961 + 0.5130 = 1.9091


Exploitabilite = 1.9091 > 0 : la strategie retransportee n'est PAS un equilibre de G.


La perte causee par l'abstraction est chiffree, des deux cotes du jeu.


Parite avec le jumeau Python : v(G), les 3 equilibres de bloc, BR_1, BR_2 et l'exploitabilite coincident.


### Lecture du resultat (exemple guide 2)

La dette de **1.9091** se decompose de facon inegale : le joueur 1 perd **1.3961** a ne pas
devier (ecart BR_1 - v), le joueur 2 seulement **0.5130**. En moyennant les duels de chaque
bloc, l'abstraction produit une strategie que **chaque** adversaire peut exploiter -- les deux
ecarts sont strictement positifs. Et le solve est exact : 6 duels originels (pour v(G)) et
3 duels abstraits resolus par enumeration de supports, aucun processus iteratif.

Les valeurs de cette cellule sont **recalculees en BCL** ; le parite avec le jumeau Python est
verifiee par assertions sur les valeurs exactes (tolerance 1e-9), pas par comparaison de texte.


## Exercice 2 -- L'atlas des paires : quelle fusion coute cher ?

La conclusion de l'exemple guide 3 le dit : la dette ne depend pas que de la **taille** de l'abstraction, mais de **quels** etats on fusionne. Verifiez-le systematiquement.

**Enonce** : pour chacune des **15 paires** {s, t} d'etats, construisez la partition a 5 blocs qui fusionne exactement cette paire (les 4 autres etats restent des singletons), resolvez-la exactement, retransportez, mesurez l'exploitabilite dans G. Affichez le classement des 15 paires par dette croissante et repondez : quelles fusions sont **gratuites** (exploitabilite nulle), laquelle coute le plus cher ?

**Sortie attendue** : un tableau `{s, t} -> exploitabilite`, trie, puis une phrase de verdict.

**Indices** :

- `# Indice` : les 15 paires s'enumerent par une double boucle `for (int s = 0; s < 6; s++) for (int t = s + 1; t < 6; t++)`.
- `# Indice` : le schema est celui de `Mj03Mesures.SolveAndTransport` dans l'exemple guide 3 -- partition, solve bloc par bloc, retransport, `Mj03Mesures.Exploitabilite`.
- `# Etape 1` : construire la partition `bloc fusionne + singletons restants` pour chaque paire.
- `# Etape 2` : collecter (paire, exploitabilite), trier, afficher.
- `# Indice` : au moins une paire devrait rendre une exploitabilite nulle -- souvenez-vous de la ligne constante de M_2.


In [5]:
// Exercice 2 : atlas des paires -- quelle fusion coute cher ? -- a completer
using System;
using System.Collections.Generic;
using System.Linq;

// TODO etudiant : pour chaque paire {s, t} parmi les 15, partition a 5 blocs fusionnant
// exactement cette paire, solve exact + retransport + exploitabilite dans G,
// puis classement des paires par dette croissante.

Console.WriteLine("Exercice a completer : l'atlas des paires (15 fusions, 15 exploitabilites).");
object resultatEx2 = null;  // TODO etudiant


Exercice a completer : l'atlas des paires (15 fusions, 15 exploitabilites).


## Exemple guide 3 -- La courbe de dette

**Ce que fait l'exemple** : faire varier la grossierete de l'abstraction et tracer la **courbe de dette** : taille du jeu abstrait (nombre d'etats abstraits) en abscisse, exploitabilite de la strategie retransportee dans le jeu d'origine en ordonnee. A chaque point : resolution **exacte** de `G_tilde`, retransport, mesure par la **meme fonction** `Mj03Mesures.Exploitabilite` qu'a l'exemple guide 2.

**Les 4 partitions forment une chaine de raffinement** -- chaque bloc d'une partition grossiere est une reunion de blocs de la partition fine :

| Point | Partition | Blocs |
|---|---|---|
| 6 etats | P6 | {0} {1} {2} {3} {4} {5} |
| 4 etats | P4 | {0,1} {2} {3} {4,5} |
| 3 etats | P3 | {0,1} {2,3} {4,5} |
| 2 etats | P2 | {0,1,2,3} {4,5} |

Sur une chaine, l'argument qualitatif tient : partition plus grossiere = strategie retransportee plus contrainte. Mais **la monotonie ne se decrete pas** -- le point a 6 etats doit rendre exactement 0 (sanity check : resoudre G puis retransporter ne change rien), et la forme des points suivants sera **nommee d'apres la sortie**, pas annoncee a l'avance.

**Sortie attendue** : un tableau de 4 points `(taille, v(G_tilde), exploitabilite)` et la forme **mesuree** nommee apres coup. La forme de cette courbe est le livrable.


In [6]:
// Exemple guide 3 : courbe de dette sur la chaine P6 < P4 < P3 < P2
using System;
using System.Globalization;
using System.Linq;

var chainPartitions = new List<(int n, int[][] blocs)>
{
    (6, new[] { new[] { 0 }, new[] { 1 }, new[] { 2 }, new[] { 3 }, new[] { 4 }, new[] { 5 } }),
    (4, new[] { new[] { 0, 1 }, new[] { 2 }, new[] { 3 }, new[] { 4, 5 } }),
    (3, new[] { new[] { 0, 1 }, new[] { 2, 3 }, new[] { 4, 5 } }),
    (2, new[] { new[] { 0, 1, 2, 3 }, new[] { 4, 5 } }),
};

// Verification de la chaine : chaque bloc de la partition fine est inclus
// dans un bloc de la partition grossiere qui la suit.
Console.WriteLine("Verification de la chaine de raffinement :");
for (int i = 0; i + 1 < chainPartitions.Count; i++)
{
    var fine = chainPartitions[i].blocs;
    var coarse = chainPartitions[i + 1].blocs;
    bool ok = fine.All(fb => coarse.Any(cb => fb.All(x => cb.Contains(x))));
    Mj03Format.That(ok, $"P{chainPartitions[i].n} ne raffine pas P{chainPartitions[i + 1].n}");
    Console.WriteLine($"  P{chainPartitions[i].n} raffine P{chainPartitions[i + 1].n} : OK");
}
Console.WriteLine();

Console.WriteLine("Courbe de dette (solve exact + retransport + exploitabilite dans G) :");
Console.WriteLine($"  {"|G_tilde|",-9}  {"v(G_tilde)",-10}  {"exploitability",-14}");
var points = new List<(int n, double v, double e)>();
foreach ((int nAbs, int[][] partition) in chainPartitions)
{
    (Func<int, int, double> s1, Func<int, int, double> s2, double vTilde) =
        Mj03Mesures.SolveAndTransport(partition);
    double e = Mj03Mesures.Exploitabilite(Mj03Jeu.Gain, s1, s2, Mj03Jeu.NStates);
    points.Add((nAbs, vTilde, e));
    Console.WriteLine($"  {nAbs,-9}  {Mj03Format.N(vTilde, 10),-10}  {Mj03Format.N(e, 14)}");
}

// Sanity check : sans abstraction, l'exploitabilite doit etre EXACTEMENT nulle.
Mj03Format.That(Math.Abs(points[0].e) < 1e-12, "P6 doit rendre 0 : solve exact + retransport identiques");

// Verdict : forme nommee D'APRES LA SORTIE.
bool nonDec = true;
for (int i = 0; i + 1 < points.Count; i++)
{
    nonDec = nonDec && points[i].e <= points[i + 1].e + 1e-9;
}
bool plateau = Math.Abs(points[1].e - points[2].e) < 1e-9;
string forme = (nonDec ? "monotone non-decroissante" : "NON monotone")
    + (plateau ? " avec un palier P4 = P3" : "");
Console.WriteLine();
Console.WriteLine($"Forme mesuree : {forme}");

// Diagnostic du palier : contribution de chaque etat a l'exploitabilite, P4 vs P3.
(Func<int, int, double> sig14, Func<int, int, double> sig24, _) = Mj03Mesures.SolveAndTransport(chainPartitions[1].blocs);
(Func<int, int, double> sig13, Func<int, int, double> sig23, _) = Mj03Mesures.SolveAndTransport(chainPartitions[2].blocs);
double[] d4 = Mj03Mesures.ContributionsParEtat(Mj03Jeu.Gain, sig14, sig24, Mj03Jeu.NStates);
double[] d3 = Mj03Mesures.ContributionsParEtat(Mj03Jeu.Gain, sig13, sig23, Mj03Jeu.NStates);
Console.WriteLine();
Console.WriteLine("Diagnostic du palier P4 = P3 (contribution de chaque etat) :");
for (int s = 0; s < Mj03Jeu.NStates; s++)
{
    Console.WriteLine($"  etat {s} : P4 -> {Mj03Format.N(d4[s], 4)}   P3 -> {Mj03Format.N(d3[s], 4)}");
}
string m2 = Mj03Jeu.MatriceLigne(2);
Console.WriteLine();
Console.WriteLine($"Explication : M_2 = {m2}, sa ligne 1 (action du joueur 1) est constante");
Console.WriteLine($"[{Mj03Jeu.Gain(2, 1, 0)}, {Mj03Jeu.Gain(2, 1, 1)}] : entre P4 et P3 la colonne retransportee pour l'etat 2");
Console.WriteLine("passe de 0 a 1, sans qu'aucun gain ne bouge -- la fusion de {2} et {3} est gratuite.");

// Parite avec le jumeau Python : les 4 points de la courbe et le diagnostic par etat.
Mj03Format.Close(points[0].v, -4.214285714285714, 1e-9, "v(G_tilde) en P6");
Mj03Format.Close(points[0].e, 0.0, 1e-12, "exploitabilite en P6");
Mj03Format.Close(points[1].v, -0.8863636363636365, 1e-9, "v(G_tilde) en P4");
Mj03Format.Close(points[1].e, 1.9090909090909092, 1e-9, "exploitabilite en P4");
Mj03Format.Close(points[2].v, -1.8863636363636365, 1e-9, "v(G_tilde) en P3");
Mj03Format.Close(points[2].e, 1.9090909090909092, 1e-9, "exploitabilite en P3");
Mj03Format.Close(points[3].v, -1.9342105263157894, 1e-9, "v(G_tilde) en P2");
Mj03Format.Close(points[3].e, 6.421052631578948, 1e-9, "exploitabilite en P2");
double[] attenduContrib = { 0.4545454545454545, 0.4545454545454546, 0.0, 0.0, 0.5, 0.5 };
for (int s = 0; s < Mj03Jeu.NStates; s++)
{
    Mj03Format.Close(d4[s], attenduContrib[s], 1e-9, $"contribution de l'etat {s} en P4");
    Mj03Format.Close(d3[s], attenduContrib[s], 1e-9, $"contribution de l'etat {s} en P3");
}
Console.WriteLine("Parite avec le jumeau Python : les 4 points de la courbe et les 6 contributions coincident.");


Verification de la chaine de raffinement :


  P6 raffine P4 : OK


  P4 raffine P3 : OK


  P3 raffine P2 : OK


Courbe de dette (solve exact + retransport + exploitabilite dans G) :


  |G_tilde|  v(G_tilde)  exploitability


... (12 outputs left out)

Explication : M_2 = [[2, -3], [2, 2]], sa ligne 1 (action du joueur 1) est constante


[2, 2] : entre P4 et P3 la colonne retransportee pour l'etat 2


passe de 0 a 1, sans qu'aucun gain ne bouge -- la fusion de {2} et {3} est gratuite.


Parite avec le jumeau Python : les 4 points de la courbe et les 6 contributions coincident.


## Exercice 3 -- Meme taille, autre dette

L'exemple guide 3 a mesure une **chaine de raffinement** -- chaque partition grossiere reunissait des blocs de la fine. Rien n'oblige une abstraction de vivre sur cette chaine.

**Enonce** : construisez une partition **P3B** de 3 blocs hors de la chaine (par exemple {0,4} {1,5} {2,3}), resolvez-la exactement, retransportez, mesurez l'exploitabilite dans G, et comparez a celle de P3 (1.9091). Meme taille de jeu abstrait, meme taille seulement ? Concluez en une phrase.

**Sortie attendue** : les 3 duels abstraits de P3B, son exploitabilite dans G, et la comparaison avec P3.

**Indices** :

- `# Indice` : les fonctions generiques (`Mj03Abstraction.AbstractMatrix`, `Mj03Mesures.Solve2x2`, `Mj03Mesures.Exploitabilite`) marchent pour **toute** partition -- la chaine n'etait qu'un choix.
- `# Etape 1` : poser `partition3b = new[] { new[] { 0, 4 }, new[] { 1, 5 }, new[] { 2, 3 } }`.
- `# Etape 2` : meme schema que l'exemple guide 3 -- solve bloc par bloc, retransport, mesure dans G.
- `# Etape 3` : la comparaison se fait a **taille egale** (3 etats abstraits dans les deux cas).


In [7]:
// Exercice 3 : une partition de taille 3 hors de la chaine -- a completer
using System;

int[][] partition3b = { new[] { 0, 4 }, new[] { 1, 5 }, new[] { 2, 3 } };  // 3 blocs, hors chaine de raffinement

// TODO etudiant : solve exact de G_tilde(P3B), retransport vers G, mesure de
// l'exploitabilite, comparaison avec P3 (1.9091) et verdict en une phrase.

Console.WriteLine("Exercice a completer : la dette de P3B contre celle de P3, a taille egale.");
object resultatEx3 = null;  // TODO etudiant


Exercice a completer : la dette de P3B contre celle de P3, a taille egale.


## Exemple guide 4 -- Le miroir forme : la courbe n'est pas une loi

La conclusion de l'exemple guide 3 posait la frontiere : ce notebook mesure SA chaine, il ne
decrete pas une loi. Le depot en possede la preuve formelle : le module Lean
`game_theory_lean/Abstraction` (ecrit comme statut formel de ce notebook) certifie un
contre-exemple complet a la generalisation naturelle « raffiner une abstraction ne peut pas
aggraver la dette retransportee ».

Sur trois duels d'entiers en strategies pures, le module construit la chaine :

| Partition | Blocs | Dette retransportee |
|---|---|---|
| grossiere Q | {0,1,2} | **3** |
| raffinee P | {0}, {1,2} | **4** |
| discrete D | {0}, {1}, {2} | **0** |

Raffiner Q en P **aggrave strictement** la dette (de 3 a 4) -- le mouvement inverse de la
forme monotone mesuree plus haut sur nos six etats -- puis la partition discrete retombe a
0. Les selles des blocs non triviaux y sont prouvees **uniques** : l'aggravation ne depend
d'aucun choix de selle.

Ce carnet est un carnet C# **autonome** : il ne lance aucun processus externe (ni compilateur
Lean, ni solveur, ni reseau) et n'importe rien du module Lean. Ce que la source Python delegue
au compilateur est ici **refait en BCL**.

La provenance formelle est donc **CITEE** : la sortie du compilateur Lean produite par le
jumeau Python sur le module `game_theory_lean/Abstraction` -- le theorem nomme, ses axioms, et
les trois valeurs reduites par le noyau -- est reproduite telle quelle dans la cellule
suivante, et marquee comme donnee de reference. Ce n'est pas un resultat produit par ce carnet.

Les fonctions que le carnet consomme dans le module -- `gain`, `vHaut`, `vBas`, `dette`,
`sommeBloc`, `detteBloc` -- sont des fonctions pures sur des entiers : elles sont
**reimplementees en BCL** dans la cellule suivante. Le carnet rejoue les duels abstraits des
blocs, puis les dettes retransportees, obtient de son cote 3, 4 et 0, et **confronte** ses
trois valeurs aux valeurs citees : un ecart ferait echouer la cellule. La lecture qui suit
porte donc sur deux chemins vers les memes nombres, dont l'un est autonome.


In [8]:
// Exemple guide 4 : le miroir forme -- la preuve formelle CITEE, la dette rejouee en BCL.
//
// Motif : ce carnet est un carnet C# AUTONOME -- il ne lance aucun processus externe
// (ni compilateur Lean, ni solveur, ni reseau) et n'importe rien du module Lean.
// Ce que la source Python delegue au compilateur est ici REFait en BCL.
//
// (1) La provenance formelle est CITEE : la sortie du compilateur Lean produite par le
//     jumeau Python sur le module `game_theory_lean/Abstraction` (theorem nomme, axioms,
//     valeurs reduites par le noyau) est reproduite ci-dessous telle quelle, et marquee
//     comme donnee de reference. Ce n'est pas un resultat produit par ce carnet.
//
// (2) Les fonctions que ce carnet consomme dans le module -- `gain`, `vHaut`, `vBas`,
//     `dette`, `sommeBloc`, `detteBloc` -- sont des fonctions PURES sur des entiers.
//     Elles sont REIMPLEMENTEES ici en BCL (classe `Mj03Abstraction`) : duels abstraits
//     des blocs, puis dettes retransportees. Les trois valeurs ainsi obtenues sont
//     comparees aux valeurs citees : les deux chemins se confrontent, aucun ne remplace
//     l'autre, et l'ecart eventuel ferait echouer la cellule.

using System;
using System.Collections.Generic;

Console.WriteLine("--- Provenance formelle CITEE (sortie du compilateur Lean, produite par le jumeau Python) ---");
Console.WriteLine("--- donnee de reference : ce carnet ne lance aucun processus externe et n'importe rien du module ---");
const string PreuveCitee = @"
raffinement_aggrave_la_dette : ∀ (tQ t0 tP : Strat),
  EstSelle d0 t0 →
    (∀ (i j : Bool), EstSelle blocQ (i, j) → (i, j) = tQ) →
      (∀ (i j : Bool), EstSelle blocP (i, j) → (i, j) = tP) →
        detteBloc [d0, d1, d2] tQ < dette d0 t0 + detteBloc [d1, d2] tP
'raffinement_aggrave_la_dette' depends on axioms: [propext]
3
4
0
raffinement_P_strict : ¬Raffine partQ partP
raffinement_D_strict : ¬Raffine partP partD
";
Console.WriteLine(PreuveCitee);

// Controle de la provenance citee : elle doit nommer le theorem, ne dependre que de
// propext, et ne porter ni sorry ni native_decide.
Mj03Format.That(PreuveCitee.Contains("raffinement_aggrave_la_dette"), "la provenance citee doit nommer le theorem");
Mj03Format.That(PreuveCitee.Contains("depends on axioms: [propext]"), "la preuve citee doit dependre seulement de propext");
Mj03Format.That(!PreuveCitee.Contains("sorry"), "aucun sorry dans la provenance citee");
Mj03Format.That(!PreuveCitee.Contains("native_decide"), "aucun native_decide dans la provenance citee");
Console.WriteLine("Provenance citee verifiee : theorem nomme, axioms = [propext], aucun sorry, aucun native_decide.");

// --- Rejeu BCL des fonctions consommees dans le module, puis confrontation avec les valeurs citees ---
Console.WriteLine();
Console.WriteLine("--- Rejeu BCL des fonctions consommees dans le module (chemin autonome) ---");

int[] blocQ = Mj03Abstraction.Somme(Mj03Abstraction.D0, Mj03Abstraction.D1, Mj03Abstraction.D2);
int[] blocP = Mj03Abstraction.Somme(Mj03Abstraction.D1, Mj03Abstraction.D2);
Mj03Format.That(blocQ[0] == 1 && blocQ[1] == 0 && blocQ[2] == -1 && blocQ[3] == 0,
    "duel abstrait du bloc grossier {0,1,2} : (1, 0, -1, 0)");
Mj03Format.That(blocP[0] == 0 && blocP[1] == 0 && blocP[2] == -1 && blocP[3] == 1,
    "duel abstrait du bloc raffine {1,2} : (0, 0, -1, 1)");
Console.WriteLine($"bloc Q {{0,1,2}} : duel abstrait ({blocQ[0]}, {blocQ[1]}, {blocQ[2]}, {blocQ[3]})");
Console.WriteLine($"bloc P {{1,2}}   : duel abstrait ({blocP[0]}, {blocP[1]}, {blocP[2]}, {blocP[3]})");

// Les trois dettes, dans l'ordre exact des trois `#eval` du temoin Lean : Q, P, D.
int detteQ = Mj03Abstraction.DetteBloc(
    new int[][] { Mj03Abstraction.D0, Mj03Abstraction.D1, Mj03Abstraction.D2 }, 0, 1);
int detteP = Mj03Abstraction.Dette(Mj03Abstraction.D0, 0, 1)
            + Mj03Abstraction.DetteBloc(
                new int[][] { Mj03Abstraction.D1, Mj03Abstraction.D2 }, 0, 0);
int detteD = Mj03Abstraction.Dette(Mj03Abstraction.D0, 0, 1)
            + Mj03Abstraction.Dette(Mj03Abstraction.D1, 1, 0)
            + Mj03Abstraction.Dette(Mj03Abstraction.D2, 0, 1);
Console.WriteLine($"dette Q (grossiere) : {detteQ}");
Console.WriteLine($"dette P (rafinee)   : {detteP}");
Console.WriteLine($"dette D (discrete)  : {detteD}");
Mj03Format.That(detteQ == 3, "dette retransportee de la partition grossiere : 3");
Mj03Format.That(detteP == 4, "dette retransportee de la partition raffinee : 4");
Mj03Format.That(detteD == 0, "dette de la partition discrete : 0");

// Confrontation : les valeurs citees (reduites par le noyau Lean sur le jumeau Python)
// contre le rejeu BCL de ce carnet. Tolerance declaree : 0 -- les grandeurs sont entieres.
List<int> citees = Mj03Format.Entiers(PreuveCitee);
Mj03Format.That(citees.Count == 3, "la provenance citee doit porter exactement les trois dettes");
int ecartQ = Math.Abs(citees[0] - detteQ);
int ecartP = Math.Abs(citees[1] - detteP);
int ecartD = Math.Abs(citees[2] - detteD);
Mj03Format.That(ecartQ == 0, "rejeu BCL et valeur citee coincident sur la dette grossiere");
Mj03Format.That(ecartP == 0, "rejeu BCL et valeur citee coincident sur la dette raffinee");
Mj03Format.That(ecartD == 0, "rejeu BCL et valeur citee coincident sur la dette discrete");
Console.WriteLine($"Ecarts (tolerance declaree : 0, entiers) : Q {ecartQ}, P {ecartP}, D {ecartD}");
Console.WriteLine($"Concordance etablie : valeurs citees ({citees[0]}, {citees[1]}, {citees[2]})"
    + $" = rejeu BCL ({detteQ}, {detteP}, {detteD}).");


// Les fonctions du module `Abstraction` que ce carnet consomme, rejouees en BCL.
// Le module les definit sur des entiers (`Int`) : la traduction est litterale, et
// sert de chemin de verification independant du noyau.
public static class Mj03Abstraction
{
    // Duels des trois etats, dans l'ordre (ligne 0, colonne 0), (ligne 0, colonne 1),
    // (ligne 1, colonne 0), (ligne 1, colonne 1).
    public static readonly int[] D0 = { 1, 0, 0, -1 };
    public static readonly int[] D1 = { -2, 0, 0, 1 };
    public static readonly int[] D2 = { 2, 0, -1, 0 };

    // `gain M i j` : gain du joueur ligne pour le profil (i, j) dans le duel M.
    public static int Gain(int[] m, int i, int j)
        => i == 0 ? (j == 0 ? m[0] : m[1]) : (j == 0 ? m[2] : m[3]);

    // `vHaut M j` : meilleure reponse ligne contre l'action colonne j.
    public static int VHaut(int[] m, int j) => Math.Max(Gain(m, 0, j), Gain(m, 1, j));

    // `vBas M i` : meilleure defense colonne contre l'action ligne i.
    public static int VBas(int[] m, int i) => Math.Min(Gain(m, i, 0), Gain(m, i, 1));

    // `dette M s` : exploitabilite du profil pur (i, j) = vHaut M j - vBas M i.
    public static int Dette(int[] m, int i, int j) => VHaut(m, j) - VBas(m, i);

    // `sommeBloc` : somme coordonnee par coordonnee de plusieurs duels.
    public static int[] Somme(params int[][] duels)
    {
        int[] somme = { 0, 0, 0, 0 };
        foreach (int[] duel in duels)
        {
            for (int k = 0; k < 4; k++)
            {
                somme[k] += duel[k];
            }
        }
        return somme;
    }

    // `detteBloc` : dette retransportee d'une strategie (i, j) dans un bloc d'etats.
    public static int DetteBloc(int[][] bloc, int i, int j)
    {
        int total = 0;
        foreach (int[] duel in bloc)
        {
            total += Dette(duel, i, j);
        }
        return total;
    }
}


--- Provenance formelle CITEE (sortie du compilateur Lean, produite par le jumeau Python) ---


--- donnee de reference : ce carnet ne lance aucun processus externe et n'importe rien du module ---



raffinement_aggrave_la_dette : ∀ (tQ t0 tP : Strat),
  EstSelle d0 t0 →
    (∀ (i j : Bool), EstSelle blocQ (i, j) → (i, j) = tQ) →
      (∀ (i j : Bool), EstSelle blocP (i, j) → (i, j) = tP) →
        detteBloc [d0, d1, d2] tQ < dette d0 t0 + detteBloc [d1, d2] tP
'raffinement_aggrave_la_dette' depends on axioms: [propext]
3
4
0
raffinement_P_strict : ¬Raffine partQ partP
raffinement_D_strict : ¬Raffine partP partD



Provenance citee verifiee : theorem nomme, axioms = [propext], aucun sorry, aucun native_decide.


--- Rejeu BCL des fonctions consommees dans le module (chemin autonome) ---


bloc Q {0,1,2} : duel abstrait (1, 0, -1, 0)


... (2 outputs left out)

dette P (rafinee)   : 4


dette D (discrete)  : 0


Ecarts (tolerance declaree : 0, entiers) : Q 0, P 0, D 0


Concordance etablie : valeurs citees (3, 4, 0) = rejeu BCL (3, 4, 0).


### Lecture du resultat (exemple guide 4)

Le bloc affiche en tete est la **provenance citee** : la sortie du compilateur Lean sur le
module `Abstraction`, produite par le jumeau Python. On y lit `raffinement_aggrave_la_dette`
dans son etat attendu -- pour tout choix de selles, la dette grossiere est strictement plus
petite que la dette raffinee -- et la ligne d'axiomes : la preuve ne depend que de `propext`
(extensionnalite propositionnelle du coeur), sans `sorry`, sans `sorryAx`, sans
`native_decide`. Les trois valeurs 3, 4, 0 sont les valeurs reduites par le noyau Lean. Ce
bloc est une donnee de reference : ce carnet ne l'a pas produite, il la cite.

Le bloc suivant rejoue en BCL les fonctions que le carnet consomme dans le module -- duels
abstraits (1, 0, -1, 0) et (0, 0, -1, 1), puis dettes retransportees 3, 4, 0 -- et la cellule
**confronte** les deux chemins : le rejeu BCL et les valeurs citees doivent donner les memes
trois nombres (tolerance declaree : 0, les grandeurs etant entieres), sinon la cellule echoue.
C'est la garantie la plus forte qu'on puisse demander ici : que la reimplementation BCL ne
derive pas du formalisme qu'elle verifie.

La lecon fixe la frontiere entre mesurer et decreter : sur nos six etats, la courbe de dette
etait monotone avec un palier ; sur trois autres duels d'entiers, un pas de raffinement
aggrave strictement la dette. La non-croissance n'est pas une propriete de la construction
« abstraire puis retransporter » -- c'est une propriete des donnees. C'est exactement ce
que la Conclusion appelle « ce notebook mesure SA chaine, il ne decrete pas une loi », et
le module Abstraction en apporte la verification du cote formel.


## Exercice 4 -- Ou la dette grossiere se concentre

La dette de la partition grossiere (3) ne se repartit pas uniformement entre les trois
etats sous la selle retransportee `stratBlocQ`. Avec le meme motif de temoin que
l'exemple guide 4, retrouver :

1. la dette que l'etat 1 paye seul sous la selle du bloc grossiere (les deux autres
   etats payant 0 chacun) ;
2. la dette que l'etat 1, puis l'etat 2, paient sous la selle `stratBlocP` du bloc
   raffine {1, 2} ;
3. verifier sur ces valeurs que la somme raffinee explique le 4 du contre-exemple.


In [9]:
// Exercice 4 : ou la dette grossiere se concentre -- a completer
using System;

// TODO etudiant : la dette de chaque etat se lit dans la REIMPLEMENTATION BCL des
// fonctions du module -- `Mj03Abstraction.Dette(duel, i, j)` pour un duel seul,
// `Mj03Abstraction.DetteBloc(bloc, i, j)` pour un bloc. Rappel : la dette
// grossiere totale vaut 3.
// 1. dette de l'etat 1 sous la selle grossiere (bloc {0,1,2}, selle (0, 1))
// 2. dette de l'etat 1, puis de l'etat 2, sous la selle raffinee (bloc {1,2}, selle (0, 0))
// Repondre en une phrase : ou se concentre la dette grossiere, et que change le
// raffinement ?

Console.WriteLine("Exercice a completer : la concentration de la dette, etat par etat.");
object resultatEx4 = null;  // TODO etudiant


Exercice a completer : la concentration de la dette, etat par etat.


***


## Conclusion : la dette de representation

L'abstraction n'est pas un raccourci gratuit : c'est un **emprunt** que l'agent fait sur la
qualite de la solution. La courbe de dette chiffre cet emprunt (solve exact partout, meme
fonction `exploitabilite`, mesure dans G) :

| Taille abstraite | v(G_tilde) | Exploitabilite dans G |
|------------------|------------|----------------------|
| 6 (pas d'abstraction) | -4.2143 | **0.0000** (sanity check exact) |
| 4 | -0.8864 | 1.9091 |
| 3 | -1.8864 | 1.9091 |
| 2 | -1.9342 | 6.4211 |

**La forme mesuree : monotone non-decroissante, avec un palier P4 = P3.** Le palier n'est pas
un artefact -- le diagnostic par etat l'explique : la dette se concentre sur les blocs {0,1}
(0.4545 par etat) et {4,5} (0.5000 par etat), tandis que la fusion {2},{3} -> {2,3} est
**gratuite** (la ligne 1 de M_2 est constante : la colonne retransportee change sans qu'aucun
gain ne bouge). Lecon operationnelle : la dette ne depend pas que de la **taille** de
l'abstraction, mais de **quels** etats on fusionne. Deux abstractions de meme taille peuvent
avoir des dettes tres differentes -- et sur une autre chaine, ou une autre graine, un
decrochement local (non-monotonie) est possible ; ce notebook mesure SA chaine, il ne decrete
pas une loi. L'exemple guide 4 en donne la preuve formelle : le module `Abstraction` certifie un decrochement strict (de 3 a 4) sur trois autres duels d'entiers.

Noter aussi la colonne v(G_tilde) : la valeur du jeu abstrait derive elle aussi (-0.89 pour
P4, contre v(G) = -4.21) -- une preuve de plus que la qualite d'une abstraction se mesure
**dans le jeu d'origine** (critere d'acceptation de l'exemple guide 2), jamais dans l'abstrait.

## La frontiere honnete

Les bornes theoriques de Kroer & Sandholm 2014, 2016 -- exploitabilite apres abstraction
bornee par un terme en O(sqrt(kappa)) ou O(sqrt(N)) selon le contexte -- sont **RAPPORTEES**,
pas redemontrees. Le notebook mesure son cas precis (fusion d'etats, zero-sum, 6 etats, solve
exact) : 4 points sur UNE chaine de raffinement et UNE graine. C'est une mesure, pas une
preuve de borne ; et la courbe est non-decroissante ICI, sur cette chaine -- le palier P4 = P3
rappelle que la stricte monotonie n'est pas garantie en general.

```
NOTION PORTEUSE : Exploitabilite <= ε(alpha, rho, ...).    (Kroer-Sandholm, rapportee)
NOTION MESUREE  : Exploitabilite_observee = f(|G_tilde|).        (ce notebook, 4 points)
```

Le notebook ne pretend pas que `f` est lineaire en `1/|G_tilde|` ; il observe la tendance sur
4 points, ce qui est une preuve de tendance, pas une preuve de borne.

## Ce que ce jumeau doit a son jumeau Python

Une parite honnete s'ecrit, elle ne se presume pas. Ici :

- les 24 gains de G sont **cites** (le MT19937 de Python n'est pas reconstructible en BCL) ;
  tous les invariants de ces donnees sont neanmoins verifies a l'execution ;
- l'abstraction, le solve, le retransport, l'exploitabilite et la courbe sont **recalcules**
  en BCL, et confrontes aux valeurs exactes du jumeau Python par assertions (tolerance 1e-9) ;
- l'exemple guide 4 **cite** la sortie du compilateur Lean sur le module `Abstraction`
  (toolchain epingle par le depot, produite par le jumeau Python) comme donnee de reference ;
  le carnet ne prouve rien de ce cote-la, il rejoue en BCL les trois dettes que cette preuve
  porte (3 / 4 / 0) et verifie qu'elles tombent juste.

## Suite (hors scope ce notebook, questions ouvertes)

- **Compagnon Lean** : la non-decroissance sur chaine de raffinement observee ici merite un
  statut formel -- la prouver sur un domaine jouable (2 etats, 2 partitions), ou y exhiber un
  contre-exemple. La question est tranchee par le depot : le module Lean `Abstraction` exhibe le contre-exemple (exemple guide 4).
- **Compagnon de la strate 7** : la question "qu'ai-je le droit d'OUBLIER" presuppose un
  critere deja choisi. La strate 7 demande : "comment APPARAIT un critere qui n'existait pas
  encore ?" -- voir `GameTheory-03d-Deux-Especes-de-Fleches-Python`.

***

**Refs** : #12229 · Kroer & Sandholm 2014, 2016 (abstraction et bornes de qualite) · Brown & Sandholm 2017 (safe subgame solving) · Vervaeke #11488 (extraire les primitives)


## References

- **Kroer, C., Sandholm, T. (2014).** "Extensive-Form Game Abstraction With Bounds." *Proceedings of the 15th ACM Conference on Economics and Computation (EC '14)*, 621-638. doi:10.1145/2600057.2602905 -- le cadre theorique qui donne des bornes de qualite de solution apres abstraction pour tout jeu a forme extensive a perfect recall : nouvelle notion de transport des strategies abstraites vers le jeu d'origine, et raffinement d'equilibre pour l'analyse. C'est le cadre quantitatif que la Conclusion rapporte en NOTION PORTEUSE (Exploitabilite <= epsilon) ; sans lui, l'affirmation que f(|G_tilde|) reste exploitable n'a aucun ancrage theorique.
- **Kroer, C., Sandholm, T. (2015).** "Limited Lookahead in Imperfect-Information Games." *Proceedings of the 24th International Joint Conference on Artificial Intelligence (IJCAI 2015)*, 575-581. Version etendue : *Artificial Intelligence* 283, article 103218, 2020 -- pas une borne d'abstraction : le papier etudie comment agir face a un adversaire dont le lookahead est borne (axes : profondeur, information de l'adversaire, tie-breaking), avec des resultats de complexite (cas polynomiaux, cas PPAD- et NP-durs). La dette vue par l'autre bout : borne de PROFONDEUR DE CALCUL, non fusion d'etats. Complement de lecture pour la question du carnet ; la Conclusion ne s'appuie pas sur lui.
- **Kroer, C., Sandholm, T. (2016).** "Imperfect-Recall Abstractions with Bounds in Games." *Proceedings of the 16th ACM Conference on Economics and Computation (EC '16)*, 459-476. doi:10.1145/2940716.2940736 -- les bornes de qualite lorsque l'abstraction abandonne meme le perfect recall (fusion d'ensembles d'information du meme joueur). C'est le referent exact du « 2016 » que la Conclusion cite a cote du 2014 -- et le cas limite de la question du carnet : jusqu'ou peut-on OUBLIER, memoire comprise, en gardant une garantie chiffree ?
- **Brown, N., Sandholm, T. (2017).** "Safe and Nested Subgame Solving for Imperfect-Information Games." *Advances in Neural Information Processing Systems 30 (NIPS 2017)*. Preprint : arXiv:1705.02955 -- le safe subgame solving : raffiner la strategie de base dans des sous-jeux atteints avec la garantie de ne pas degrader l'exploitabilite de la strategie de depart, y compris face aux actions adverses hors abstraction (resolution imbriquee en cascade dans l'arbre) ; composant de Libratus. Le pendant operationnel de la dette du carnet : l'abstraite est un point de depart qu'on ameliore localement sans jamais perdre la garantie.
- **Johanson, M., Bard, N., Lanctot, M., Gibson, R., Bowling, M. (2012).** "Efficient Nash Equilibrium Approximation through Monte Carlo Counterfactual Regret Minimization." *Proceedings of the 11th International Conference on Autonomous Agents and Multiagent Systems (AAMAS 2012)*, 837-846 -- MCCFR : l'echantillonnage Monte Carlo qui rend la minimisation du regret contrefactaire praticable dans les grands jeux. Le pendant algorithmique des bornes de Kroer-Sandholm, et la famille du solveur du carnet (CFR, prerequis carnet 13).
- **Sandholm, W. H. (2010).** *Population Games and Evolutionary Dynamics.* MIT Press. ISBN 978-0-262-19587-4 -- le socle des jeux de population et des dynamiques d'apprentissage vers l'equilibre : le cadre dans lequel la mesure du carnet (distance en gain espere a l'equilibre) s'interprete. NB : William H. Sandholm (economiste, Wisconsin), distinct de Tuomas Sandholm (CMU) co-auteur des entrees precedentes -- deux auteurs du meme nom de famille.
- **Vervaeke, J., Lillicrap, T. P., Richards, B. A. (2012).** "Relevance Realization and the Emerging Framework in Cognitive Science." *Journal of Logic and Computation* 22(1), 79-99. doi:10.1093/logcom/exp067 -- la relevance realization : comment un critere de pertinence peut emerger de l'interaction systeme-environnement plutot que d'etre pose d'avance. Eclairage pour la question ouverte du carnet (« comment APPARAIT un critere qui n'existait pas encore ? », Compagnon de la strate 7). La mention inline (Vervaeke #11488) renvoie a une discussion interne CoursIA ; la formulation academique est ce papier.


## À retenir

- **L'abstraction est un emprunt, pas un raccourci gratuit.** La dette retransportée (exploitabilité mesurée dans le jeu d'origine `G`) chiffre la qualité perdue : 0 sans abstraction, jusqu'à `6.42` pour la plus grossière.
- **La dette ne dépend pas que de la *taille* de l'abstraction, mais de *quels* états on fusionne.** Deux abstractions de même taille peuvent avoir des dettes très différentes : le palier `P4 = P3` est « gratuit » car la fusion `{2},{3} → {2,3}` ne fait bouger aucun gain.
- **La non-croissance de la dette est une propriété des *données*, pas de la construction.** Sur nos six états la courbe est monotone, mais le module Lean `Abstraction` certifie un contre-exemple où un pas de raffinement *aggrave* strictement la dette (3 → 4). Ce notebook mesure SA chaîne, il ne décréte pas une loi.
- **Un jumeau honnète écrit ses frontières.** Ici : les gains de `G` sont cités (le générateur du jumeau Python n'est pas reconstructible en BCL), le reste est recalculé, et l'exemple guide 4 cite la sortie du compilateur Lean — les fonctions du module y sont réimplémentées en BCL, pas réimportées.

## Vérifiez votre compréhension

1. Pourquoi le palier `P4 = P3` est-il « gratuit » ? Quel fait sur la matrice `M_2` l'explique ?
2. Pourquoi la courbe de dette est-elle monotone non-décroissante sur nos six états, alors que l'exemple guide 4 exhibe un raffinement qui l'aggrave ?
3. Que certifie le module Lean `Abstraction` que ce carnet C# (BCL seul, aucun processus externe) ne peut pas prouver lui-même ?

<details>
<summary>Éléments de réponse</summary>

1. Le palier est gratuit parce que la fusion `{2},{3} -> {2,3}` n'ajoute aucune dette : la courbe donne la même exploitabilité `1.9091` pour `|G_tilde| = 4` et `|G_tilde| = 3`, et le diagnostic par état est identique dans les deux partitions, les états 2 et 3 contribuant `+0.0000` chacun (cellule 12). Le fait qui l'explique est dans `M_2 = [[2, -3], [2, 2]]` : sa ligne 1 (action du joueur 1) est constante `[2, 2]`, si bien qu'entre P4 et P3 la colonne retransportée pour l'état 2 passe de 0 à 1 sans qu'aucun gain ne bouge (sortie de la cellule 12, repris dans la conclusion). La dette reste concentrée sur les blocs `{0,1}` (0.4545 par état) et `{4,5}` (0.5000 par état).

2. La forme monotone est une propriété des données de cette chaîne, pas de la construction « abstraire puis retransporter ». Sur nos six états (graine 42, gains cités), les quatre points mesurés sont `0.0000, 1.9091, 1.9091, 6.4211` : le raffinement P4 -> P3 ne coûte rien parce que la ligne 1 de `M_2` est constante, et seule la partition la plus grossière (`|G_tilde| = 2`) fait monter la dette à `6.4211`. L'exemple guide 4 porte sur trois autres duels d'entiers : là, un pas de raffinement aggrave strictement la dette, les valeurs réduites par le noyau Lean étant `3` puis `4` (cellule 16). Il n'y a donc pas de contradiction : le carnet C# mesure SA chaîne, sur une graine et quatre points, il ne décréte pas une loi ; la non-croissance n'est pas une propriété de la construction, elle est une propriété des données (cellules 17 et 21).

3. Le module Lean certifie un énoncé universel sur les trois duels témoins, quantifié sur tous les choix de selles : `raffinement_aggrave_la_dette : ∀ (tQ t0 tP : Strat), EstSelle d0 t0 → (∀ (i j : Bool), EstSelle blocQ (i, j) → (i, j) = tQ) → (∀ (i j : Bool), EstSelle blocP (i, j) → (i, j) = tP) → detteBloc [d0, d1, d2] tQ < dette d0 t0 + detteBloc [d1, d2] tP`, avec `'raffinement_aggrave_la_dette' depends on axioms: [propext]` — aucun `sorry`, aucun `sorryAx`, aucun `native_decide`. Il certifie aussi la stricte monotonie de la chaîne dans les deux sens : `raffinement_P_strict : ¬Raffine partQ partP` et `raffinement_D_strict : ¬Raffine partP partD` (sortie de la cellule 16, lecture de la cellule 17). Le noyau C# ne peut ni énoncer ni prouver un théorème Lean : il peut seulement soumettre un témoin au compilateur (`lean`) et lire son verdict, comme le fait la cellule 16 ; sa propre mesure reste un échantillon fini de quatre points sur une chaîne et une graine, sans universalité.

</details>

## Pour aller plus loin

- **Kroer & Sandholm (2014, 2015, 2016)** — les bornes de qualité de solution après abstraction (transport des stratégies, bounded lookahead, imperfect recall).
- **Le module Lean `game_theory_lean/Abstraction`** — le statut formel de ce notebook : il certifie le contre-exemple de non-monotonie.
- La distinction entre **borne de profondeur de calcul** (bounded lookahead) et **fusion d'états** (abstraction).
